## 01 — Live Layer Switching

Everything comes together here.

We wire three things into one map:
1. **LOD selection** — `get_lod(zoom)` picks which dataset to use
2. **Grid index culling** — the correct grid index is queried for the current viewport
3. **Event handling** — zoom and pan events trigger the right updates

After this notebook, we have a working map that automatically serves the right level of detail for any zoom and any viewport location.

## Setup — Load One File and Build In-Memory LOD Indexes

This fixed version uses only `data/ne_10m_railroads.geojson`.

It does **not** require `../../data/lod/railroads_coarse.geojson`, `railroads_medium.geojson`, `railroads_fine.geojson`, or `railroads_extra_fine.geojson`. Instead, it derives four LOD feature sets in memory from the one raw railroad file.

In [1]:
import copy
import json
import math
import time
from pathlib import Path

# This version uses ONLY the one railroad file you already have:
# data/ne_10m_railroads.geojson
# It does NOT read or create ../../data/lod/railroads_*.geojson files.

def find_raw_railroad_file():
    """Search this notebook folder and its parents for data/ne_10m_railroads.geojson."""
    cwd = Path.cwd().resolve()
    for base in [cwd] + list(cwd.parents):
        candidate = base / "data" / "ne_10m_railroads.geojson"
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        "Could not find data/ne_10m_railroads.geojson. "
        "Keep the notebook inside the project folder that contains the data folder."
    )

RAW_PATH = find_raw_railroad_file()

print("Loading single railroad file:")
print(" ", RAW_PATH)

with open(RAW_PATH) as f:
    raw_data = json.load(f)

raw_features = raw_data["features"]
print(f"Loaded original file: {len(raw_features):,} features")


def iter_lines(geometry):
    """Yield coordinate lists for LineString or MultiLineString geometries."""
    coords = geometry.get("coordinates", [])
    geom_type = geometry.get("type")

    if geom_type == "LineString":
        yield coords
    elif geom_type == "MultiLineString":
        for part in coords:
            yield part


def point_distance_to_segment(p, a, b):
    px, py = p[0], p[1]
    ax, ay = a[0], a[1]
    bx, by = b[0], b[1]

    dx = bx - ax
    dy = by - ay

    if dx == 0 and dy == 0:
        return math.hypot(px - ax, py - ay)

    t = ((px - ax) * dx + (py - ay) * dy) / (dx * dx + dy * dy)
    t = max(0, min(1, t))

    proj_x = ax + t * dx
    proj_y = ay + t * dy

    return math.hypot(px - proj_x, py - proj_y)


def douglas_peucker(points, epsilon):
    """Simple Douglas-Peucker line simplification for lon/lat coordinate lists."""
    if len(points) <= 2:
        return points

    start = points[0]
    end = points[-1]

    max_dist = -1
    max_index = 0

    for i in range(1, len(points) - 1):
        dist = point_distance_to_segment(points[i], start, end)
        if dist > max_dist:
            max_dist = dist
            max_index = i

    if max_dist > epsilon:
        left = douglas_peucker(points[:max_index + 1], epsilon)
        right = douglas_peucker(points[max_index:], epsilon)
        return left[:-1] + right

    return [start, end]


def simplify_geometry(geometry, epsilon):
    """Return a simplified copy of a LineString or MultiLineString geometry."""
    geom_type = geometry.get("type")
    coords = geometry.get("coordinates", [])

    if geom_type == "LineString":
        return {
            "type": "LineString",
            "coordinates": douglas_peucker(coords, epsilon),
        }

    if geom_type == "MultiLineString":
        return {
            "type": "MultiLineString",
            "coordinates": [douglas_peucker(part, epsilon) for part in coords if len(part) >= 2],
        }

    return copy.deepcopy(geometry)


def scalerank(feature):
    try:
        return int(feature.get("properties", {}).get("scalerank", 99))
    except Exception:
        return 99


def make_lod_features(features, epsilon, max_scalerank=None):
    """Create an in-memory LOD feature list from the single raw file."""
    out = []
    for feature in features:
        if max_scalerank is not None and scalerank(feature) > max_scalerank:
            continue

        new_feature = copy.deepcopy(feature)
        new_feature["geometry"] = simplify_geometry(new_feature["geometry"], epsilon)
        out.append(new_feature)

    return out

# Four LOD levels are created in memory only. No new files are written.
LOD_SETTINGS = {
    "coarse":     {"epsilon": 1.00, "max_scalerank": 4},
    "medium":     {"epsilon": 0.50, "max_scalerank": 6},
    "fine":       {"epsilon": 0.15, "max_scalerank": None},
    "extra_fine": {"epsilon": 0.03, "max_scalerank": None},
}

print("\nBuilding in-memory LOD feature sets from the single file...")
lod_features = {}
for name, setting in LOD_SETTINGS.items():
    t0 = time.perf_counter()
    lod_features[name] = make_lod_features(
        raw_features,
        epsilon=setting["epsilon"],
        max_scalerank=setting["max_scalerank"],
    )
    elapsed = time.perf_counter() - t0
    print(f"  {name:<12} {len(lod_features[name]):>6,} features  {elapsed:.2f}s")


Loading single railroad file:
  /workspaces/ricardoayala2510-Spatial-Data-Mapping/assigments completed/03-Data_Manager/data/ne_10m_railroads.geojson
Loaded original file: 25,413 features

Building in-memory LOD feature sets from the single file...
  coarse        2,845 features  1.02s
  medium        5,946 features  3.08s
  fine         25,413 features  6.19s
  extra_fine   25,413 features  7.47s


In [2]:
def iter_points(coords):
    """Yield individual [lon, lat] points from nested GeoJSON coordinates."""
    if not coords:
        return

    first = coords[0]
    if isinstance(first, (int, float)):
        yield coords
    else:
        for part in coords:
            yield from iter_points(part)


def feature_bbox(feature):
    points = list(iter_points(feature["geometry"]["coordinates"]))
    if not points:
        return [0, 0, 0, 0]

    lons = [p[0] for p in points]
    lats = [p[1] for p in points]
    return [min(lons), min(lats), max(lons), max(lats)]


class GridIndex:
    def __init__(self, cell_size=10.0):
        self.cell_size = cell_size
        self.cells = {}

    def _cells_for_bbox(self, bbox):
        lon_min, lat_min, lon_max, lat_max = bbox
        col_min = int((lon_min + 180) / self.cell_size)
        col_max = int((lon_max + 180) / self.cell_size)
        row_min = int((lat_min +  90) / self.cell_size)
        row_max = int((lat_max +  90) / self.cell_size)
        return [
            (col, row)
            for col in range(col_min, col_max + 1)
            for row in range(row_min, row_max + 1)
        ]

    def build(self, features):
        self.cells = {}
        for idx, feature in enumerate(features):
            for cell in self._cells_for_bbox(feature_bbox(feature)):
                if cell not in self.cells:
                    self.cells[cell] = []
                self.cells[cell].append((idx, feature))

    def query(self, viewport_bbox):
        seen = set()
        results = []
        for cell in self._cells_for_bbox(viewport_bbox):
            for idx, feature in self.cells.get(cell, []):
                if idx not in seen:
                    seen.add(idx)
                    results.append(feature)
        return results


In [3]:
print("Building grid indexes...")
lod_indexes = {}
for name, features in lod_features.items():
    t0 = time.perf_counter()
    idx = GridIndex(cell_size=10.0)
    idx.build(features)
    elapsed = time.perf_counter() - t0
    lod_indexes[name] = idx
    print(f"  {name:<12} built in {elapsed:.3f}s")

print("\nAll indexes ready.")

Building grid indexes...
  coarse       built in 0.016s
  medium       built in 0.081s
  fine         built in 0.521s
  extra_fine   built in 0.162s

All indexes ready.


## The Decision and Utility Functions

In [4]:
def get_lod(zoom):
    z = int(math.floor(zoom))
    if z <= 3:  return "coarse"
    if z <= 6:  return "medium"
    if z <= 10: return "fine"
    return "extra_fine"


def leaflet_bounds_to_bbox(bounds):
    (lat_min, lon_min), (lat_max, lon_max) = bounds
    return [lon_min, lat_min, lon_max, lat_max]

## The Live Map

Two event handlers wire everything together:

- `on_zoom_change` — fires when zoom changes, switches to the correct LOD index, then re-queries
- `on_bounds_change` — fires when the user pans, re-queries the current LOD index

Both update the same layer, so there is only ever one GeoJSON layer on the map.

In [5]:
from ipyleaflet import Map, GeoJSON
import ipywidgets as widgets

# ── state ────────────────────────────────────────────────────────────────────
current_lod = get_lod(5)   # start at zoom 5

# ── map setup ────────────────────────────────────────────────────────────────
m = Map(center=[48.5, 10.0], zoom=5)

layer = GeoJSON(
    data={"type": "FeatureCollection", "features": []},
    style={"color": "#cc3300", "weight": 1.5, "opacity": 0.8},
    name="Railroads"
)
m.add(layer)

# ── status widgets ────────────────────────────────────────────────────────────
lod_label     = widgets.Label(value="LOD: —")
feature_label = widgets.Label(value="Features: —")
time_label    = widgets.Label(value="Query: —")
status_bar    = widgets.HBox([lod_label, feature_label, time_label])

# ── update function ───────────────────────────────────────────────────────────
def update(*args):
    global current_lod

    if not m.bounds:
        return

    # Decide which LOD to use
    new_lod = get_lod(m.zoom)
    if new_lod != current_lod:
        current_lod = new_lod

    # Query the correct grid index
    vp = leaflet_bounds_to_bbox(m.bounds)
    t0 = time.perf_counter()
    visible = lod_indexes[current_lod].query(vp)
    elapsed_ms = (time.perf_counter() - t0) * 1000

    # Update the layer
    layer.data = {"type": "FeatureCollection", "features": visible}

    # Update status
    lod_label.value     = f"LOD: {current_lod}"
    feature_label.value = f"  Features: {len(visible):,}"
    time_label.value    = f"  Query: {elapsed_ms:.2f}ms"

# ── wire events ───────────────────────────────────────────────────────────────
m.observe(update, names=["zoom", "bounds"])
update()   # initial render

widgets.VBox([m, status_bar])

**Try it:**
- Zoom out to 2 — the LOD status switches to `coarse` and feature count drops
- Zoom into a city — the LOD switches to `fine` or `extra_fine` and feature count drops (culling)
- Pan around at a fixed zoom — feature count updates as different regions come into view

The query time in the status bar shows how fast each lookup is.

## What We Just Built

Let's be explicit about the system:

```
User interaction
      │
      ▼
zoom / pan event
      │
      ├──► get_lod(zoom)  ──────► select correct GridIndex
      │
      └──► leaflet_bounds_to_bbox(bounds)
                │
                ▼
          index.query(viewport_bbox)
                │
                ▼
          visible features (deduplicated)
                │
                ▼
          update GeoJSON layer
```

Every component was built from scratch across these five modules:
- The in-memory LOD feature sets derived from the single railroad file (Module 02 idea)
- The bounding box intersection test (Module 03)
- The grid spatial index (Module 04)
- The zoom decision function (Module 05)

## Exercise A

Add a **zoom indicator** to the status bar that shows the current zoom level and a simple text label of the geographic scale (e.g. `zoom 5 — country scale`).

Use the zoom-to-scale table from Notebook 00 as a guide.

In [6]:
# Copy the map setup from above and add a zoom level + scale label to the status bar
from ipyleaflet import Map, GeoJSON
import ipywidgets as widgets


def zoom_scale_label(zoom):
    """Return a simple text description for the current zoom level."""
    z = int(math.floor(zoom))
    if z <= 3:
        return "world / continent scale"
    elif z <= 6:
        return "country scale"
    elif z <= 10:
        return "city / regional scale"
    else:
        return "street / detail scale"


# ── state ────────────────────────────────────────────────────────────────────
current_lod_a = get_lod(5)

# ── map setup ────────────────────────────────────────────────────────────────
m_a = Map(center=[48.5, 10.0], zoom=5)

layer_a = GeoJSON(
    data={"type": "FeatureCollection", "features": []},
    style={"color": "#cc3300", "weight": 1.5, "opacity": 0.8},
    name="Railroads"
)
m_a.add(layer_a)

# ── status widgets ────────────────────────────────────────────────────────────
zoom_label_a    = widgets.Label(value="Zoom: —")
lod_label_a     = widgets.Label(value="LOD: —")
feature_label_a = widgets.Label(value="Features: —")
time_label_a    = widgets.Label(value="Query: —")
status_bar_a    = widgets.HBox([zoom_label_a, lod_label_a, feature_label_a, time_label_a])

# ── update function ───────────────────────────────────────────────────────────
def update_a(*args):
    global current_lod_a

    if not m_a.bounds:
        return

    # Decide which LOD to use
    new_lod = get_lod(m_a.zoom)
    if new_lod != current_lod_a:
        current_lod_a = new_lod

    # Query the correct grid index
    vp = leaflet_bounds_to_bbox(m_a.bounds)
    t0 = time.perf_counter()
    visible = lod_indexes[current_lod_a].query(vp)
    elapsed_ms = (time.perf_counter() - t0) * 1000

    # Update the layer
    layer_a.data = {"type": "FeatureCollection", "features": visible}

    # Update status
    zoom_label_a.value    = f"Zoom: {m_a.zoom} — {zoom_scale_label(m_a.zoom)}"
    lod_label_a.value     = f"  LOD: {current_lod_a}"
    feature_label_a.value = f"  Features: {len(visible):,}"
    time_label_a.value    = f"  Query: {elapsed_ms:.2f}ms"

# ── wire events ───────────────────────────────────────────────────────────────
m_a.observe(update_a, names=["zoom", "bounds"])
update_a()

widgets.VBox([m_a, status_bar_a])


## Exercise B

The `update()` function is called for both zoom and bounds changes — a single event handler covers both.

This means if the user zooms AND pans at the same time (which ipyleaflet reports as two rapid events), `update()` runs twice. The second call sees the final state and overwrites the first — so the result is correct, but redundant work was done.

Add a `print` statement inside `update()` that shows which property triggered the call (`zoom` or `bounds`). Then scroll/zoom the map and observe the sequence. Do zoom and bounds always fire together?

In [7]:
# Copy the map and add print(change['name']) inside update() to trace events
from ipyleaflet import Map, GeoJSON
import ipywidgets as widgets

# ── state ────────────────────────────────────────────────────────────────────
current_lod_b = get_lod(5)

# ── map setup ────────────────────────────────────────────────────────────────
m_b = Map(center=[48.5, 10.0], zoom=5)

layer_b = GeoJSON(
    data={"type": "FeatureCollection", "features": []},
    style={"color": "#3366cc", "weight": 1.5, "opacity": 0.8},
    name="Railroads trace"
)
m_b.add(layer_b)

# ── status widgets ────────────────────────────────────────────────────────────
lod_label_b     = widgets.Label(value="LOD: —")
feature_label_b = widgets.Label(value="Features: —")
time_label_b    = widgets.Label(value="Query: —")
status_bar_b    = widgets.HBox([lod_label_b, feature_label_b, time_label_b])

# ── update function with event tracing ────────────────────────────────────────
def update_b(change=None):
    global current_lod_b

    if change is None:
        print("update triggered by: initial render")
    else:
        print(f"update triggered by: {change['name']}")

    if not m_b.bounds:
        return

    # Decide which LOD to use
    new_lod = get_lod(m_b.zoom)
    if new_lod != current_lod_b:
        current_lod_b = new_lod

    # Query the correct grid index
    vp = leaflet_bounds_to_bbox(m_b.bounds)
    t0 = time.perf_counter()
    visible = lod_indexes[current_lod_b].query(vp)
    elapsed_ms = (time.perf_counter() - t0) * 1000

    # Update the layer
    layer_b.data = {"type": "FeatureCollection", "features": visible}

    # Update status
    lod_label_b.value     = f"LOD: {current_lod_b}"
    feature_label_b.value = f"  Features: {len(visible):,}"
    time_label_b.value    = f"  Query: {elapsed_ms:.2f}ms"

# ── wire events ───────────────────────────────────────────────────────────────
m_b.observe(update_b, names=["zoom", "bounds"])
update_b()

widgets.VBox([m_b, status_bar_b])


update triggered by: initial render


## Check Your Understanding

The map derives **four** LOD feature sets from one raw file and builds **four** grid indexes at startup — one per LOD level. This takes a few seconds and uses memory.

An alternative design would build the index lazily — only when the user first reaches that zoom range. Describe the tradeoffs between eager (build all at startup) and lazy (build on first use) index construction for this specific application.

---

**Answer:** Eager construction means the notebook spends more time and memory at startup because it builds all four indexes immediately. The benefit is that layer switching is smooth later: when the user zooms into a new LOD range, the correct index is already ready, so the map can update quickly.

Lazy construction makes startup faster and uses less memory at first because it only builds the index for the LOD level that is actually needed. The downside is that the first time the user reaches a new zoom range, the map may pause while that index is built. For this application, eager loading is better if smooth interaction is the priority, while lazy loading is better if the dataset is very large or if many users will only view one or two zoom ranges.


## Next

In [Module 06 — Putting It All Together](../06-Putting_It_Together/README.md), we assemble a clean, well-organized viewer notebook and reflect on what we built — before seeing the library version in Module 07.